In [1]:
# obligatory imports
import sys
sys.path.append("../")

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

import joblib

from dataset.data_classes import PytorchSpectraDataset
from dataset.dataset_builder import SpectralDatasetBuilder

from sklearn.neighbors import NearestNeighbors

from utils.utils import load_config
from utils.experiments import generate_batch_spectra, load_models, load_colors

In [2]:
# Load data info, scalers and the categorical encoder
label_encoder = joblib.load("../pretrained_models/scalers/label_encoder.pkl")
minmax_scaler = joblib.load("../pretrained_models/scalers/minmax_scaler.pkl")
categorical_conditions = joblib.load("../pretrained_models/scalers/categorical_conditions.pkl")
continuous_conditions = joblib.load("../pretrained_models/scalers/continuous_conditions.pkl")
standard_scaler = joblib.load("../pretrained_models/scalers/std_scaler.pkl")

c:\Users\Kerschbaumer\anaconda3\envs\py\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Kerschbaumer\anaconda3\envs\py\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Load models
models = load_models()

In [4]:
dataset_builder = SpectralDatasetBuilder(data_path='../data/in-silico-gen-data.parquet', data_split={"train": 0.8, "test": 0.2}, config_path='../config.yaml')
dataset = PytorchSpectraDataset("../data/h4h_raw_2024-09-16_with_subject_ids.parquet")
column_names = dataset.df_test_scaled.iloc[:,:519].columns.values.astype(float)

In [38]:
df_train_real = dataset.df_train_scaled
df_train_real.columns = np.hstack([column_names, df_train_real.columns.values[519:]], dtype=object)
df_test_real = dataset.df_test_scaled
df_test_real.columns = np.hstack([column_names, df_test_real.columns.values[519:]], dtype=object)

In [6]:
synthetic_datasets = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models = load_models()

cfg = load_config('../config.yaml')
continous_conditions = cfg["condition_labels"]["continuous"]
categorical_conditions = cfg["condition_labels"]["categorical"]
conditions = categorical_conditions + continous_conditions

In [7]:
GAN_FILE = "../data/CBEGAN_generated.parquet"
VAE_FILE = "../data/CVAE_generated.parquet"
DIFF_FILE = "../data/CDIFF_generated.parquet"

df_vae = pd.read_parquet(VAE_FILE)
df_gan = pd.read_parquet(GAN_FILE)
df_diff = pd.read_parquet(DIFF_FILE)

In [8]:
def _nn_distance(query, reference, metric, same_set):
    """Distance from each row of `query` to its nearest row in `reference`.
 
    If `same_set` is True, the first neighbor is the point itself (distance 0),
    so we take the second-nearest instead.
    """
    k = 2 if same_set else 1
    nn = NearestNeighbors(n_neighbors=k, metric=metric).fit(reference)
    dist, _ = nn.kneighbors(query)
    return dist[:, -1]

# Overall

In [9]:
nnd_cvae_test = _nn_distance(df_vae[column_names.astype(str)].values, df_train_real[column_names].values, metric='euclidean', same_set=False)
nnd_cbegan_test = _nn_distance(df_gan[column_names.astype(str)].values, df_train_real[column_names].values, metric='euclidean', same_set=False)
nnd_cdiff_test = _nn_distance(df_diff[column_names.astype(str)].values, df_train_real[column_names].values, metric='euclidean', same_set=False)
nnd_real = _nn_distance(df_test_real[column_names].values, df_train_real[column_names].values, metric='euclidean', same_set=False)

In [10]:
nnaa_cvae = sum(nnd_real > nnd_cvae_test) / len(nnd_cdiff_test)
nnaa_cbegan = sum(nnd_real > nnd_cbegan_test) / len(nnd_cdiff_test)
nnaa_cdiff = sum(nnd_real > nnd_cdiff_test) / len(nnd_cdiff_test)

In [11]:
print('CVAE: ', nnaa_cvae)
print('CBEGAN: ', nnaa_cbegan)
print('CDIFF: ', nnaa_cdiff)

CVAE:  0.446266297905966
CBEGAN:  0.5962070327933623
CDIFF:  0.47036744369814304


# SVC

In [40]:
nnd_real = _nn_distance(df_test_real[column_names].values, df_train_real[column_names].values, metric='euclidean', same_set=False)

result = pd.DataFrame()

sim_datasets = {
    'CVAE': df_vae,
    'CBEGAN': df_gan,
    'CDIFF': df_diff,
}

for model_name, df_sim in sim_datasets.items():
    svc_nnaas = []
    for i, SVC in df_sim.groupby('test_set'):
        nnd_sim = _nn_distance(SVC[column_names.astype(str)].values, df_train_real[column_names].values, metric='euclidean', same_set=False)
        test_set_idx = SVC['test_set'].values[0]
        nnd_real = _nn_distance(df_test_real[df_test_real['test_set']==test_set_idx][column_names].values, df_train_real[column_names].values, metric='euclidean', same_set=False)
        nnaa = sum(nnd_real > nnd_sim) / len(nnd_real)
        svc_nnaas.append(nnaa)

    result[model_name] = svc_nnaas

In [41]:
result

,CVAE,CBEGAN,CDIFF
0,0.448521,0.610651,0.467456
1,0.424674,0.577699,0.442467
2,0.459075,0.610913,0.499407
3,0.456702,0.599051,0.467378
4,0.451957,0.583630,0.462633
5,0.436686,0.595266,0.482840
